In [0]:
# OPENIA
%pip install databricks-openai>=0.3.0
%pip install openai>=1.12.0


INFO: pip is looking at multiple versions of pyjwt[crypto] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-openai[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-client to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 24.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 47.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 51.9 MB/s  0:00:0

In [0]:
#import urllib.request
#import os

# URL do ficheiro no repositório de dados do Databricks Academy
#url = "https://gist.githubusercontent.com/chodimella/4b9d05f31e62f058e3fb7ecc8c3b2ab0/raw/airbnb-listings.csv"

# Criar volume no Unity Catalog (se não existir)
#spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.airbnb_data")

# Caminho de destino num Volume do Unity Catalog (serverless-compatible)
#target_path = "/Volumes/workspace/default/airbnb_data/airbnb-cleaned-mlflow.csv"

# Descarregar o CSV diretamente via HTTP e escrever no Volume
#response = urllib.request.urlopen(url)
#csv_content = response.read()

#with open(target_path, "wb") as f:
#    f.write(csv_content)

#print(f"Ficheiro guardado com sucesso em: {target_path}")

In [0]:

# Load the Airbnb dataset from the Unity Catalog Volume
airbnb_path = "/Volumes/workspace/default/airbnb_data/airbnb-cleaned-mlflow.csv"
airbnb_df = spark.read.csv(airbnb_path, header=True, inferSchema=True)

display(airbnb_df.limit(5))

id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
10080,D1 - Million Dollar View 2 BR,30899,Rami,null,Downtown,49.28771582115245,-123.12110274713065,Entire home/apt,295,60,16,2017-02-26,0.18,31,161
11400,Central Lovely Rm in Victorian Home,42305,Jaynne,null,Riley Park,49.25375606647688,-123.11265899043721,Private room,60,364,70,2016-06-14,0.64,1,365
13188,Garden level studio in ideal loc.,51466,Family Guns,null,Riley Park,49.245770071782864,-123.10515815885476,Entire home/apt,119,2,165,2019-01-07,1.51,1,269
13357,! Wow! 2bed 2bath 1bed den Harbour View Apartment!,52118,Liliana,null,Downtown Eastside,49.27954754717459,-123.10545719258528,Entire home/apt,151,30,56,2018-08-31,0.51,2,210
13358,Urban Boutique Suite heart of Downtown Vancouver!,52116,Lynn,null,West End,49.28208988905561,-123.1251498273402,Entire home/apt,119,1,384,2019-01-21,3.65,1,21


In [0]:
import os
#from google.cloud import bigquery
import pandas as pd
import logging

In [0]:
import pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession
logger = logging.getLogger()

In [0]:
import pyspark.sql.functions as F
Analise_df = airbnb_df.select(
    F.col("neighbourhood_group"),
    F.col("last_review"),
    F.col("price")    
    )


In [0]:
Analise_df.count()

4873

User settings

Developer

Settings => Developer => Geneate new token, para simular criei para 2 dias apenas

In [0]:
#DATABRICKS_HOST = 'https://numero....gcp.databricks.com/' # seu databricks
DATABRICKS_HOST = 'https://dbc-0b69ef93-5247.cloud.databricks.com/'
# meu: token databricks ia
#DATABRICKS_TOKEN_FABI ="token do databricks" # Ai, genie, all api
DATABRICKS_TOKEN_FABI ="SEU TOKEN"
SUMMARIZER_ENDPOINT = 'databricks-gpt-oss-120b'
# SUMMARIZER_ENDPOINT = 'databricks-gpt-5-nano'
# SUMMARIZER_ENDPOINT = 'databricks-gpt-oss-20b'

In [0]:

from databricks.sdk import WorkspaceClient
from databricks.sdk.core import Config


In [0]:
# 🔒 Configuração "fixa" para PAT: sem perfis, sem auto-descoberta
cfg = Config(
    host=DATABRICKS_HOST,
    token=DATABRICKS_TOKEN_FABI,
    auth_type="pat",
    profile=None,           # não carregar ~/.databrickscfg
)

# Cria o cliente do workspace com PAT APENAS
w = WorkspaceClient(config=cfg)

In [0]:
Analise_df.show()

+-------------------+-----------+-----+
|neighbourhood_group|last_review|price|
+-------------------+-----------+-----+
|               NULL| 2017-02-26|  295|
|               NULL| 2016-06-14|   60|
|               NULL| 2019-01-07|  119|
|               NULL| 2018-08-31|  151|
|               NULL| 2019-01-21|  119|
|               NULL| 2018-09-30|  146|
|               NULL| 2018-07-08|  140|
|               NULL| 2013-07-13|  140|
|               NULL| 2018-08-26|  220|
|               NULL| 2018-02-16| 1201|
|               NULL| 2018-03-31|  135|
|               NULL| 2019-01-03|   98|
|               NULL| 2018-10-31|   55|
|               NULL| 2019-02-02|   80|
|               NULL| 2019-01-02|   94|
|               NULL| 2018-09-07|  108|
|               NULL| 2019-01-17|  110|
|               NULL| 2019-01-02|  230|
|               NULL| 2019-01-29|   50|
|               NULL| 2018-12-27|   66|
+-------------------+-----------+-----+
only showing top 20 rows


In [0]:
Analise_df_pd= Analise_df.toPandas()
Analise_df_pd.head()



,neighbourhood_group,last_review,price
0,None,2017-02-26,295
1,None,2016-06-14,60
2,None,2019-01-07,119
3,None,2018-08-31,151
4,None,2019-01-21,119


In [0]:
Analise_df_pd['neighbourhood_group'] = Analise_df_pd['neighbourhood_group'].fillna('Simple Houses')
Analise_df_pd['neighbourhood_group'].unique()

array(['Simple Houses', 'Kitsilano', 'Hastings-Sunrise',
       'Kensington-Cedar Cottage', 'Downtown Eastside', 'Downtown',
       'Oakridge', 'West End', 'Dunbar Southlands', 'Peter & Linda',
       'Marpole'], dtype=object)

In [0]:
Analise_df_pd['neighbourhood_group'].unique()

array(['Simple Houses', 'Kitsilano', 'Hastings-Sunrise',
       'Kensington-Cedar Cottage', 'Downtown Eastside', 'Downtown',
       'Oakridge', 'West End', 'Dunbar Southlands', 'Peter & Linda',
       'Marpole'], dtype=object)

In [0]:
Analise_df_pd['last_review'].unique()

array(['2017-02-26', '2016-06-14', '2019-01-07', '2018-08-31',
       '2019-01-21', '2018-09-30', '2018-07-08', '2013-07-13',
       '2018-08-26', '2018-02-16', '2018-03-31', '2019-01-03',
       '2018-10-31', '2019-02-02', '2019-01-02', '2018-09-07',
       '2019-01-17', '2019-01-29', '2018-12-27', None, '0.09',
       '2018-10-11', '2019-01-27', '2018-09-06', '2016-08-22',
       '2013-06-20', '2019-01-14', '2018-10-01', '2018-05-23',
       '2019-01-20', '2018-10-07', '2018-09-10', '2019-02-01',
       '2018-12-31', '2018-12-09', '2018-12-15', '2019-01-01',
       '2019-01-05', '2018-04-04', '2019-01-08', '2018-12-25',
       '2018-04-09', '2018-09-15', '2018-10-03', '2014-09-04',
       '2019-01-06', '2019-02-03', '2018-09-23', '2018-11-13',
       '2018-10-28', '2018-07-27', '2018-02-28', '2018-03-30',
       '2018-09-18', '2019-01-31', '2018-09-04', '2019-01-04',
       '2018-12-16', '2018-06-07', '2018-09-03', '2018-09-01',
       '2018-08-03', '2018-08-16', '2014-08-31', '2018-

In [0]:
# Converter 'last_review' para datetime, removendo linhas que não são datas válidas
Analise_df_pd['last_review'] = pd.to_datetime(Analise_df_pd['last_review'], errors='coerce')

# Manter apenas linhas em que 'last_review' é uma data válida (descartar NaT)
Analise_df_pd = Analise_df_pd[Analise_df_pd['last_review'].notna()].reset_index(drop=True)

Analise_df_pd['last_review'].unique()

<DatetimeArray>
['2017-02-26 00:00:00', '2016-06-14 00:00:00', '2019-01-07 00:00:00',
 '2018-08-31 00:00:00', '2019-01-21 00:00:00', '2018-09-30 00:00:00',
 '2018-07-08 00:00:00', '2013-07-13 00:00:00', '2018-08-26 00:00:00',
 '2018-02-16 00:00:00',
 ...
 '2018-06-22 00:00:00', '2018-05-02 00:00:00', '2018-07-15 00:00:00',
 '2018-07-01 00:00:00', '2018-05-25 00:00:00', '2018-06-14 00:00:00',
 '2018-06-19 00:00:00', '2018-06-10 00:00:00', '2018-09-20 00:00:00',
 '2018-07-13 00:00:00']
Length: 448, dtype: datetime64[ns]

In [0]:
# Converter 'price' (texto) para numérico antes de somar
Analise_df_pd['price'] = pd.to_numeric(Analise_df_pd['price'], errors='coerce')

In [0]:
Analise_df_pd.loc[Analise_df_pd['neighbourhood_group'] == 'Downtown']

,neighbourhood_group,last_review,price


In [0]:
Analise_df_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4130 entries, 0 to 4129
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   neighbourhood_group  4130 non-null   object        
 1   last_review          4130 non-null   datetime64[ns]
 2   price                4130 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 96.9+ KB


In [0]:


# Pivot: soma do preço por neighbourhood_group
# (sem 'values=', o pivot usava 'last_review' (datas), que não suportam soma)
Analise_df_pivot_pd = Analise_df_pd.pivot_table(
    index='neighbourhood_group',
    values='price',
    aggfunc='sum',
    fill_value=0
).sort_index().reset_index(drop=False)
Analise_df_pivot_pd.head()

,neighbourhood_group,price
0,Simple Houses,658059


In [0]:

# Pivot: soma do preço por neighbourhood_group
# (sem 'values=', o pivot usava 'last_review' (datas), que não suportam soma)
Analise_df_pivot_pd = Analise_df_pd.pivot_table(
    index='last_review',
    values='price',
    aggfunc='sum',
    fill_value=0
).sort_index().reset_index(drop=False)
Analise_df_pivot_pd.head()

,last_review,price
0,2013-06-02,180
1,2013-06-20,98
2,2013-07-13,140
3,2013-09-17,62
4,2013-09-30,366


index='last_review': Agrupa os dados usando a coluna last_review como as linhas da tabela.

values='price': Define que a coluna a ser calculada/agregada é price.

aggfunc='sum': Aplica a função de soma aos valores de price para cada data de last_review.

fill_value=0: Substitui eventuais valores ausentes (NaN) resultantes do pivô por 0.

In [0]:
Analise_df_pivot_pd = Analise_df_pd.pivot_table(
    index='last_review',
    columns='neighbourhood_group',
    values='price',
    aggfunc='sum',
    fill_value=0
).sort_index().reset_index(drop=False)
Analise_df_pivot_pd.head()

neighbourhood_group,last_review,Simple Houses
0,2013-06-02,180
1,2013-06-20,98
2,2013-07-13,140
3,2013-09-17,62
4,2013-09-30,366


In [0]:
# usar o prompt
prompt_usuario = (
    "Com base nos indicadores abaixo, gere uma análise executiva em português do Brasil. "
    "Evite jargões técnicos, seja direto e claro. Use emojis para tornar a mensagem mais interativa. "
    "Separe os raciocínios em parágrafos curtos. Destaque tendências, variações críticas e recomendações. "
    "Não utilize o termo 'ontem' nem 'hoje' para referenciar a data. "
    "Diga apenas a data referente. "
    "Essa análise é a categoria do imovel : 'neighbourhood_group'. "
    "Todos os valores são sobre data e o price (preço).\n\n"
    + Analise_df_pd.to_string()
)

In [0]:


# System message adaptada
system_msg = (
    "Você é um analista de dados especializado em gerar insights estratégicos com base em indicadores de negócio de preços de casas o Airbnb"

    "Escreva resumos claros e objetivos em português do Brasil, com linguagem acessível e sem jargões técnicos excessivos. "

    "Sempre traga junto da informação um texto explicativo, não apenas os números. "

    "Destaque todos os pontos de quedas e de crescimento no período analisado, citando a performance de todas as colunas."

    "Use o tamanho necessário de frases para analisar todos os períodos, separando os raciocínios em parágrafos curtos para facilitar a leitura. "

    "Evite caracteres tipográficos sofisticados; prefira ASCII quando possível. "

    "Utilize emojis para tornar a conversa mais interativa e amigável. "

    "Responda apenas com o texto final, não mostre seu raciocínio. "

    "Resuma para no máximo 5 parágrafos. "

    "Para todos os valores numéricos, destaque de alguma maneira. "

    "Sempre traga recomendações e Não precisa de próximos passos. "

    "Os valores das colunas com 'estão por datas' significam a quantidade de registros do local das casas (neighbourhood_group) e seus preços (price na época). "

    "Os valores de 'neighbourhood_group' indicam a categoria do local aonde fica a casa e que foi atribuido. "

    "'price' preço da casa ou moradia"

    "Deixe todas as datas no formato 'dd/mm/ano'. Sendo que os dados de entrada no sistema esta ano-mes-dia (aaaa-mm-dd) "

    "O FORMATO DAS RESPOSTAS TERIA QUE SER EM HTML"

    #"Sempre traga o valor de comparação de 'sessoes' em porcentagem, ou seja, se no forecast/previsão haverá aumento ou queda nas sessões (sessoes) se comparados com o valor real. "
)

def _extract_text_from_chat_response(response):
    return response.choices[0].message.content
    
# Envio para o agente via OpenAI-compatible client
if SUMMARIZER_ENDPOINT:
    import requests

    api_url = f"{DATABRICKS_HOST.rstrip('/')}/serving-endpoints/{SUMMARIZER_ENDPOINT}/invocations"
    headers = {
        "Authorization": f"Bearer {DATABRICKS_TOKEN_FABI}",
        "Content-Type": "application/json"
    }
    payload = {
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt_usuario}
        ],
        "temperature": 0.2,
        "max_tokens": 2000
    }

    resp = requests.post(api_url, headers=headers, json=payload)
    resp.raise_for_status()
    result = resp.json()

    text = result["choices"][0]["message"]["content"]

    print(text)


[{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': 'We need to produce an executive analysis in Portuguese, with HTML format, summarizing trends, variations, and recommendations. Use emojis, highlight numeric values, dates in dd/mm/aaaa. No jargon, max 5 paragraphs, each short. Need to highlight points of decline and growth, performance of all columns (neighbourhood_group is same "Simple Houses", last_review date, price). So we need to analyze price over time. Data huge, but we can summarize: many low prices early (e.g., 2013-2015), spikes high (e.g., 2018-2019 high values up to 1200, 1500, 2000). Identify periods of growth: 2018-2019 price increase, peaks in early 2019. Declines: some low values in 2016-2017. Provide recommendations: monitor high volatility, adjust pricing, focus on high-demand periods.\n\nWe need to format dates dd/mm/aaaa. So convert sample dates. Provide numeric values highlighted (maybe bold). Use emojis.\n\nWe need HTML output. Provide up to 5 pa

In [0]:
displayHTML(text_airbnb)

```html
 📊 Visão geral – Todos os registros pertencem ao grupo Simple Houses . As datas vão de 01/02/2013 a 05/02/2019 e os preços variam de menos de 30 até mais de 2.000 unidades monetárias. 

 🚀 Tendência de alta – Até 2015 os preços permaneciam baixos (entre 30 e 150 ). A partir de 2016‑2017 há um leve aumento, mas o verdadeiro salto ocorre em 2018, quando muitos valores ultrapassam 500 e chegam a 1.200 ‑ 2.000 . O pico de 2019 (até 02/02/2019 ) mostra vários imóveis acima de 1.000 , indicando forte valorização. 

 📉 Quedas e volatilidade – Apesar da alta geral, há quedas pontuais: em 14/06/2016 o preço foi 37 , em 22/07/2017 caiu para 39 e em 05/08/2018 registrou 45 . Esses recuos aparecem em períodos de baixa demanda ou ofertas promocionais. 

 💡 Recomendações – 
 
 Monitorar os períodos de pico (final de 2018 e início de 2019) para ajustar tarifas e maximizar receita. 
 Usar os valores baixos como oportunidades de promoção ou teste de novos pacotes. 
 Implementar um modelo de precificação dinâmico que considere a data da última revisão e a variação histórica de preços. 
 
 

 🔚 Resumo – O mercado de Simple Houses mostrou uma clara evolução de preços baixos para valores elevados entre 2018‑2019, com picos acima de 2.000 . A estratégia deve focar em aproveitar os momentos de alta, enquanto controla a volatilidade nos períodos de queda. 
```

In [0]:

def extract_final_text(response):
    if isinstance(response, list):
        return "\n\n".join(
            item['text'] for item in response if item.get('type') == 'text'
        )
    elif hasattr(response, 'choices'):
        return response.choices[0].message.content
    else:
        return str(response)

text_airbnb = extract_final_text(text)
print(text_airbnb)

```html
<p>📊 <strong>Visão geral</strong> – Todos os registros pertencem ao grupo <em>Simple Houses</em>. As datas vão de <strong>01/02/2013</strong> a <strong>05/02/2019</strong> e os preços variam de menos de <strong>30</strong> até mais de <strong>2.000</strong> unidades monetárias.</p>

<p>🚀 <strong>Tendência de alta</strong> – Até 2015 os preços permaneciam baixos (entre <strong>30</strong> e <strong>150</strong>). A partir de 2016‑2017 há um leve aumento, mas o verdadeiro salto ocorre em 2018, quando muitos valores ultrapassam <strong>500</strong> e chegam a <strong>1.200</strong>‑<strong>2.000</strong>. O pico de 2019 (até <strong>02/02/2019</strong>) mostra vários imóveis acima de <strong>1.000</strong>, indicando forte valorização.</p>

<p>📉 <strong>Quedas e volatilidade</strong> – Apesar da alta geral, há quedas pontuais: em <strong>14/06/2016</strong> o preço foi <strong>37</strong>, em <strong>22/07/2017</strong> caiu para <strong>39</strong> e em <strong>05/08/2018</strong

In [0]:
def color_alerta(val):
    if val is None or val == 50:
        return ''
    if 50 <= val <= 70:
        return 'background-color: #ffe6e6; color: #d32f2f;'  # vermelho claro
    elif 70 < val:
        return 'background-color: #e6f7ff; color: #1976d2;'  # azul claro
    else:
        return ''

In [0]:
Analise_df_pivot_pd.head(10)

neighbourhood_group,last_review,Simple Houses
0,2013-06-02,180
1,2013-06-20,98
2,2013-07-13,140
3,2013-09-17,62
4,2013-09-30,366
5,2013-10-13,112
6,2014-02-03,119
7,2014-05-23,279
8,2014-06-24,199
9,2014-07-02,199


In [0]:

styled_df = Analise_df_pivot_pd.reset_index(drop=True).style.applymap(color_alerta, subset=['Simple Houses'])
display(styled_df)

/home/spark-f689b54a-d2f8-4eb7-831e-3d/.ipykernel/85/command-5519959947335592-2101284205:2: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_df = Analise_df_pivot_pd.reset_index(drop=True).style.applymap(color_alerta, subset=['Simple Houses'])


neighbourhood_group,last_review,Simple Houses
0,2013-06-02 00:00:00,180
1,2013-06-20 00:00:00,98
2,2013-07-13 00:00:00,140
3,2013-09-17 00:00:00,62
4,2013-09-30 00:00:00,366
5,2013-10-13 00:00:00,112
6,2014-02-03 00:00:00,119
7,2014-05-23 00:00:00,279
8,2014-06-24 00:00:00,199
9,2014-07-02 00:00:00,199


In [0]:

styled_df_html = styled_df.to_html()


In [0]:

# COMMAND ----------

texto_final_aluguel_rapido_html =f"""
<h3> Exemplo de AI chat no Databricks com os dados do Airbnb</h3>
<br>
<br>
{text_airbnb}
<br>
<h3> Airbnb </h3>
{styled_df_html}

<br>


"""

# COMMAND ----------

displayHTML(texto_final_aluguel_rapido_html)


neighbourhood_group,last_review,Simple Houses
0,2013-06-02 00:00:00,180
1,2013-06-20 00:00:00,98
2,2013-07-13 00:00:00,140
3,2013-09-17 00:00:00,62
4,2013-09-30 00:00:00,366
5,2013-10-13 00:00:00,112
6,2014-02-03 00:00:00,119
7,2014-05-23 00:00:00,279
8,2014-06-24 00:00:00,199
9,2014-07-02 00:00:00,199
